# ChromaDB Parent-Child Chunk Health Check

Validates the integrity of parent-child chunk relationships stored in ChromaDB.

This notebook performs:
- Connection and collection verification
- Child and parent chunk enumeration
- Orphan detection (children without parents, parents without children)
- Metadata consistency validation
- Relationship health scoring
- Detailed remediation recommendations

⚠️ **IMPORTANT**: If you've recently run ingestion with `--reset`, restart the kernel before running this notebook to ensure you're seeing fresh data and not cached results from previous sessions. Use: **Kernel → Restart Kernel**

## 0. Import Required Libraries

## Configuration (Optional)

Adjust these settings to control analysis depth and prevent hangs on large collections.

In [ ]:
# Configuration settings to prevent hangs on large collections

# Maximum chunks to analyse (set to None for unlimited, or a number like 100000)
MAX_CHUNKS_TO_ANALYSE = None  # None = analyse all chunks

# Skip expensive debug operations if collection is large
SKIP_DEBUG_ON_LARGE_COLLECTIONS = True  # Set to False to always run debug cell
LARGE_COLLECTION_THRESHOLD = 50000  # Collections larger than this are considered "large"

# Batch size for child ID verification (smaller = slower but safer)
BATCH_SIZE = 100

# Progress indicator frequency (show progress every N chunks)
PROGRESS_EVERY = 10000

print("✓ Configuration loaded")
print(f"  Max chunks to analyse: {MAX_CHUNKS_TO_ANALYSE or 'Unlimited'}")
print(f"  Skip debug on large collections: {SKIP_DEBUG_ON_LARGE_COLLECTIONS}")
print(f"  Large collection threshold: {LARGE_COLLECTION_THRESHOLD:,}")


In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from chromadb import PersistentClient
from scripts.ingest.ingest_config import IngestConfig
import pandas as pd
import json
from collections import defaultdict
from typing import Dict, List, Set, Tuple, Optional

print("✓ Libraries imported successfully")

## 1. Connect to ChromaDB

In [ ]:
# Load configuration
from scripts.utils.db_factory import get_default_vector_path, get_vector_client

config = IngestConfig()

# Get the correct ChromaDB path using factory pattern
PersistentClient_class, using_sqlite = get_vector_client(prefer="chroma")
chroma_path = get_default_vector_path(Path(config.rag_data_path), using_sqlite)

print(f"rag_data_path: {config.rag_data_path}")
print(f"ChromaDB path: {chroma_path}")
print()

# Connect to ChromaDB
client = PersistentClient_class(path=str(chroma_path))

print(f"✓ Connected to ChromaDB at: {chroma_path}")
print()

# List available collections
available_collections = client.list_collections()
print(f"Available collections ({len(available_collections)}):")
for col in available_collections:
    print(f"  - {col.name} (count: {col.count()})")

print()

# Try to get chunk collection if it exists
try:
    chunk_collection = client.get_collection(config.chunk_collection_name)
    print(f"✓ Chunk collection: {config.chunk_collection_name}")
    print(f"✓ Collection count: {chunk_collection.count()}")
except Exception as e:
    print(f"⚠ Chunk collection not found: {config.chunk_collection_name}")
    print(f"  Error: {e}")
    chunk_collection = None

In [ ]:
# Verify fresh connection and display collection info
from datetime import datetime

if chunk_collection is None:
    print("⚠ Chunk collection not available - skipping analysis")
    print("  Please ingest data first using: make ingest")
else:
    print(f"⏰ Analysis timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\n📊 Current collection state:")
    print(f"  Collection name: {config.chunk_collection_name}")
    print(f"  Total documents: {chunk_collection.count():,}")
    
    # Check if collection appears to be freshly reset

## 2. Query Child Entries

In [ ]:
# Retrieve all chunks - FRESH QUERY with progress indicator
# Note: This queries ChromaDB directly, not from cached variables

print(f"🔄 Fetching fresh data from ChromaDB...")
total_count = chunk_collection.count()
print(f"📊 Collection has {total_count:,} total chunks")

# Safety check: warn if collection is very large
if total_count > 100000:
    print(f"\n⚠️  WARNING: Large collection detected ({total_count:,} chunks)")
    print(f"     This query may take 30+ seconds and could hang the notebook.")
    print(f"     Consider adding a limit if you just want to spot-check health.")
    print(f"     Proceeding with full query...")

# Use a reasonable limit if collection is enormous (optional safety)
MAX_CHUNKS = 500000  # Adjust based on your system's memory
if total_count > MAX_CHUNKS:
    print(f"\n🛑 Collection exceeds safety limit ({MAX_CHUNKS:,})")
    print(f"     Fetching first {MAX_CHUNKS:,} chunks only...")
    all_chunks = chunk_collection.get(
        limit=MAX_CHUNKS,
        include=["documents", "metadatas"]
    )
    print(f"⚠️  Note: Analysis based on partial data ({MAX_CHUNKS:,}/{total_count:,} chunks)")
else:
    all_chunks = chunk_collection.get(
        include=["documents", "metadatas"]
    )

print(f"✓ Retrieved {len(all_chunks['ids']):,} chunks from database\n")

# Separate child and parent chunks
child_chunks = []
parent_chunks = []
regular_chunks = []

print("🔄 Categorising chunks...")
for i, (chunk_id, metadata) in enumerate(zip(all_chunks["ids"], all_chunks["metadatas"])):
    # Progress indicator for large collections
    if i > 0 and i % 10000 == 0:
        print(f"   Processed {i:,}/{len(all_chunks['ids']):,} chunks...")
    
    chunk_data = {
        "id": chunk_id,
        "metadata": metadata,
        "is_parent": metadata.get("is_parent", False) if metadata else False,
        "chunk_type": metadata.get("chunk_type", "regular") if metadata else "regular"
    }
    
    if chunk_data["is_parent"]:
        parent_chunks.append(chunk_data)
    elif chunk_data["chunk_type"] == "child" or "parent_id" in (metadata or {}):
        child_chunks.append(chunk_data)
    else:
        regular_chunks.append(chunk_data)

print(f"\n✓ Total chunks: {len(all_chunks['ids']):,}")
print(f"  - Parent chunks: {len(parent_chunks):,}")
print(f"  - Child chunks: {len(child_chunks):,}")
print(f"  - Regular chunks: {len(regular_chunks):,}")

# Check if we're seeing document data (shouldn't exist after code ingestion with --reset)
if regular_chunks:
    sample_metadata = regular_chunks[0]['metadata'] if regular_chunks[0]['metadata'] else {}
    doc_type = sample_metadata.get('doc_type', 'unknown')
    source_type = sample_metadata.get('source_type', 'unknown')
    
    if doc_type in ['confluence', 'html', 'pdf'] or 'document' in source_type.lower():
        print(f"\n⚠️  WARNING: Detected document chunks (not code)!")
        print(f"     Doc type: {doc_type}, Source type: {source_type}")
        print(f"     If you ran code ingestion with --reset, this is unexpected.")
        print(f"     👉 Try: Restart kernel and re-run to clear cached variables")

# Display sample child chunks
if child_chunks:
    print(f"\n📋 Sample child chunks (first 5):")
    for i, chunk in enumerate(child_chunks[:5]):
        parent_id = chunk["metadata"].get("parent_id", "N/A") if chunk["metadata"] else "N/A"
        print(f"  {i+1}. ID: {chunk['id'][:60]}...")
        print(f"     Parent ID: {parent_id[:60] if parent_id != 'N/A' else 'N/A'}...")


## 3. Query Parent Entries

In [ ]:
# Build parent ID -> child IDs mapping
parent_to_children: Dict[str, Set[str]] = defaultdict(set)
child_to_parent: Dict[str, str] = {}

for child in child_chunks:
    parent_id = child["metadata"].get("parent_id") if child["metadata"] else None
    if parent_id:
        parent_to_children[parent_id].add(child["id"])
        child_to_parent[child["id"]] = parent_id

# Parent chunk analysis
print(f"📊 Parent chunk analysis:")
print(f"  - Total parent chunks stored: {len(parent_chunks)}")
print(f"  - Parents with children references: {len(parent_to_children)}")

# List parent chunks with their child counts
print(f"\n📋 Parent chunks and their children:")
for parent in parent_chunks[:10]:
    parent_id = parent["id"]
    child_count = len(parent_to_children.get(parent_id, set()))
    
    # Try to parse child_ids from metadata
    child_ids_json = parent["metadata"].get("child_ids") if parent["metadata"] else None
    if child_ids_json:
        try:
            child_ids_list = json.loads(child_ids_json)
            stored_child_count = len(child_ids_list)
            print(f"  {parent_id}")
            print(f"    - Stored child_ids in metadata: {stored_child_count}")
            print(f"    - Actual children found: {child_count}")
        except json.JSONDecodeError:
            print(f"  {parent_id} - Could not parse child_ids metadata")

In [ ]:
# DEBUG: Investigate metadata structure and missing child chunks
# ⚠️ This cell can be slow if there are many parent chunks

# Check if we should skip this cell for large collections
total_chunks = len(all_chunks['ids'])
if SKIP_DEBUG_ON_LARGE_COLLECTIONS and total_chunks > LARGE_COLLECTION_THRESHOLD:
    print(f"⏭️  SKIPPING DEBUG ANALYSIS")
    print(f"   Collection size ({total_chunks:,}) exceeds threshold ({LARGE_COLLECTION_THRESHOLD:,})")
    print(f"   Set SKIP_DEBUG_ON_LARGE_COLLECTIONS = False in config cell to force run")
    print(f"   This prevents notebook hangs on large collections.")
else:
    print("🔍 DEBUG: Analysing metadata structure and child chunk references\n")

    # Sample a few parent chunks to see their complete metadata
    print("📋 Sample parent metadata (first 3 parents):")
    for i, parent in enumerate(parent_chunks[:3]):
        print(f"\n  Parent {i+1}: {parent['id'][:60]}...")
        if parent['metadata']:
            for key, value in parent['metadata'].items():
                if len(str(value)) > 100:
                    print(f"    {key}: {str(value)[:100]}...")
                else:
                    print(f"    {key}: {value}")
        else:
            print(f"    [No metadata]")

    # Extract all child IDs referenced by parents
    all_referenced_child_ids = set()
    print("\n\n🔍 Extracting child IDs from parent metadata:")
    for parent in parent_chunks:
        parent_id = parent['id']
        child_ids_json = parent['metadata'].get('child_ids') if parent['metadata'] else None
        
        if child_ids_json:
            try:
                child_ids_list = json.loads(child_ids_json)
                all_referenced_child_ids.update(child_ids_list)
            except (json.JSONDecodeError, TypeError):
                pass

    print(f"✓ Total unique child IDs referenced: {len(all_referenced_child_ids)}")

    # Check if any of these referenced child IDs exist in the collection
    # ⚠️ This can be SLOW - using batched lookup for efficiency
    print(f"\n🔍 Checking if referenced child IDs exist in collection...")

    if len(all_referenced_child_ids) > 100:
        print(f"⏳ Large number of child IDs ({len(all_referenced_child_ids):,}) - using batch check")
        print(f"   This may take 10-30 seconds...")

    existing_child_ids = set()

    # Batch check in groups to avoid hanging
    child_id_list = list(all_referenced_child_ids)
    total_batches = (len(child_id_list) + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_num in range(total_batches):
        start_idx = batch_num * BATCH_SIZE
        end_idx = min(start_idx + BATCH_SIZE, len(child_id_list))
        batch = child_id_list[start_idx:end_idx]
        
        # Progress indicator for large batches
        if total_batches > 10 and batch_num % 10 == 0:
            print(f"   Checking batch {batch_num + 1}/{total_batches}...")
        
        try:
            result = chunk_collection.get(ids=batch, include=["metadatas"])
            if result and result['ids']:
                existing_child_ids.update(result['ids'])
        except Exception as e:
            # If batch fails, try individual lookups (slower but more reliable)
            for child_id in batch:
                try:
                    result = chunk_collection.get(ids=[child_id], include=["metadatas"])
                    if result and result['ids'] and len(result['ids']) > 0:
                        existing_child_ids.add(child_id)
                except:
                    pass

    print(f"\n✓ Referenced child IDs found in collection: {len(existing_child_ids):,}")
    print(f"✓ Referenced child IDs NOT in collection: {len(all_referenced_child_ids - existing_child_ids):,}")

    if len(all_referenced_child_ids - existing_child_ids) > 0:
        print(f"\n📋 Sample missing child IDs (first 10):")
        for child_id in list(all_referenced_child_ids - existing_child_ids)[:10]:
            print(f"  - {child_id[:80]}...")

    # Show all chunk IDs to check for naming patterns
    print(f"\n\n🔍 Analysing chunk ID patterns:")
    # Use already-loaded data instead of re-querying
    all_ids = set(all_chunks['ids'])
    print(f"  Total chunk IDs in collection: {len(all_ids):,}")
    print(f"  IDs containing '-child_': {len([id for id in all_ids if '-child_' in id]):,}") 
    print(f"  IDs containing '-parent_': {len([id for id in all_ids if '-parent_' in id]):,}")

    # Show sample of non-parent, non-child IDs
    regular_ids = [id for id in all_ids if '-parent_' not in id and '-child_' not in id]
    print(f"  IDs without parent/child markers: {len(regular_ids):,}")
    if regular_ids:
        print(f"  Sample regular IDs (first 5):")
        for rid in regular_ids[:5]:
            print(f"    - {rid[:80]}...")


## 4. Validate Parent-Child Relationships

In [ ]:
# Validate relationships
issues = {
    "orphaned_children": [],
    "orphaned_parents": [],
    "missing_parent_records": [],
    "mismatched_child_counts": []
}

# Check for orphaned children
for child in child_chunks:
    parent_id = child["metadata"].get("parent_id") if child["metadata"] else None
    
    if not parent_id:
        issues["orphaned_children"].append(child["id"])
    else:
        # Check if parent exists
        parent_exists = any(p["id"] == parent_id for p in parent_chunks)
        if not parent_exists:
            issues["missing_parent_records"].append({
                "child_id": child["id"],
                "referenced_parent_id": parent_id
            })

# Check for orphaned parents
all_parent_ids = {p["id"] for p in parent_chunks}
for parent_id in all_parent_ids:
    if parent_id not in parent_to_children or len(parent_to_children[parent_id]) == 0:
        issues["orphaned_parents"].append(parent_id)

# Check for mismatched child counts
for parent in parent_chunks:
    parent_id = parent["id"]
    child_ids_json = parent["metadata"].get("child_ids") if parent["metadata"] else None
    
    if child_ids_json:
        try:
            stored_child_ids = json.loads(child_ids_json)
            actual_child_ids = parent_to_children.get(parent_id, set())
            
            if len(stored_child_ids) != len(actual_child_ids):
                issues["mismatched_child_counts"].append({
                    "parent_id": parent_id,
                    "stored_count": len(stored_child_ids),
                    "actual_count": len(actual_child_ids)
                })
        except json.JSONDecodeError:
            pass

print("🔍 Relationship Validation Results:")
print(f"  - Orphaned children (no parent_id): {len(issues['orphaned_children'])}")
print(f"  - Orphaned parents (no children): {len(issues['orphaned_parents'])}")
print(f"  - Missing parent records: {len(issues['missing_parent_records'])}")
print(f"  - Mismatched child counts: {len(issues['mismatched_child_counts'])}")

# Show details of issues
if issues["orphaned_children"]:
    print(f"\n⚠️  Orphaned children (first 5):")
    for child_id in issues["orphaned_children"][:5]:
        print(f"  - {child_id}")

if issues["missing_parent_records"]:
    print(f"\n⚠️  Missing parent records (first 5):")
    for issue in issues["missing_parent_records"][:5]:
        print(f"  - Child {issue['child_id']} references missing parent {issue['referenced_parent_id']}")

if issues["orphaned_parents"]:
    print(f"\n⚠️  Orphaned parents (first 5):")
    for parent_id in issues["orphaned_parents"][:5]:
        print(f"  - {parent_id}")

if issues["mismatched_child_counts"]:
    print(f"\n⚠️  Mismatched child counts (first 5):")
    for issue in issues["mismatched_child_counts"][:5]:
        print(f"  - Parent {issue['parent_id']}: stored={issue['stored_count']}, actual={issue['actual_count']}")

## 5. Generate Health Check Report

In [ ]:
# Calculate health score
total_issues = sum(len(v) if isinstance(v, list) else 0 for v in issues.values())
issue_count = sum(1 for v in issues.values() if (isinstance(v, list) and len(v) > 0) or (isinstance(v, int) and v > 0))

# Health score: 100 - (issues * 2)
health_score = max(0, 100 - (total_issues * 2))

print("=" * 70)
print("📊 CHROMADB PARENT-CHILD CHUNK HEALTH CHECK REPORT")
print("=" * 70)

print(f"\n🎯 Overall Health Score: {health_score}/100", end="")
if health_score >= 90:
    print(" ✅ EXCELLENT")
elif health_score >= 70:
    print(" ⚠️  GOOD")
elif health_score >= 50:
    print(" ⚠️  FAIR")
else:
    print(" ❌ POOR")

print(f"\n📈 Collection Statistics:")
print(f"  - Total chunks: {len(all_chunks['ids'])}")
print(f"  - Parent chunks: {len(parent_chunks)}")
print(f"  - Child chunks: {len(child_chunks)}")
print(f"  - Regular chunks: {len(regular_chunks)}")

print(f"\n🔗 Relationship Statistics:")
print(f"  - Parents with children: {len(parent_to_children)}")
print(f"  - Children with parents: {len(child_to_parent)}")
if len(parent_chunks) > 0:
    avg_children_per_parent = len(child_chunks) / len(parent_chunks) if len(parent_chunks) > 0 else 0
    print(f"  - Average children per parent: {avg_children_per_parent:.1f}")

print(f"\n⚠️  Issues Found: {total_issues}")
print(f"  - Orphaned children: {len(issues['orphaned_children'])}")
print(f"  - Orphaned parents: {len(issues['orphaned_parents'])}")
print(f"  - Missing parent records: {len(issues['missing_parent_records'])}")
print(f"  - Mismatched child counts: {len(issues['mismatched_child_counts'])}")

# Recommendations
print(f"\n💡 Recommendations:")
if health_score >= 90:
    print("  ✅ Database is healthy. No action required.")
elif len(issues["orphaned_children"]) > 0:
    print(f"  ⚠️  Consider re-ingesting {len(issues['orphaned_children'])} orphaned children")
elif len(issues["orphaned_parents"]) > 0:
    print(f"  ⚠️  Consider cleaning up {len(issues['orphaned_parents'])} unused parent chunks")
elif len(issues["missing_parent_records"]) > 0:
    print(f"  ⚠️  Found {len(issues['missing_parent_records'])} broken relationships - re-ingestion recommended")
else:
    print("  ✅ No critical issues detected.")

print("\n" + "=" * 70)

## 6. Export Summary as DataFrame

In [ ]:
# Create summary dataframes for export

# Summary of chunk types
chunk_summary = pd.DataFrame({
    "Type": ["Parent", "Child", "Regular", "Total"],
    "Count": [len(parent_chunks), len(child_chunks), len(regular_chunks), len(all_chunks['ids'])],
    "Percentage": [
        f"{len(parent_chunks)/len(all_chunks['ids'])*100:.1f}%" if len(all_chunks['ids']) > 0 else "0%",
        f"{len(child_chunks)/len(all_chunks['ids'])*100:.1f}%" if len(all_chunks['ids']) > 0 else "0%",
        f"{len(regular_chunks)/len(all_chunks['ids'])*100:.1f}%" if len(all_chunks['ids']) > 0 else "0%",
        "100%"
    ]
})

# Summary of issues
issue_summary = pd.DataFrame({
    "Issue Type": ["Orphaned Children", "Orphaned Parents", "Missing Parent Records", "Mismatched Child Counts"],
    "Count": [
        len(issues["orphaned_children"]),
        len(issues["orphaned_parents"]),
        len(issues["missing_parent_records"]),
        len(issues["mismatched_child_counts"])
    ],
    "Severity": ["Medium", "Low", "High", "Medium"]
})

print("\n📋 Chunk Type Distribution:")
print(chunk_summary.to_string(index=False))

print("\n\n⚠️  Issue Summary:")
print(issue_summary.to_string(index=False))

# Parent-Child mapping summary
parent_child_summary = []
for parent in parent_chunks:
    parent_id = parent["id"]
    children = parent_to_children.get(parent_id, set())
    
    parent_child_summary.append({
        "Parent ID": parent_id,
        "Child Count": len(children),
        "Doc ID": parent["metadata"].get("doc_id", "N/A") if parent["metadata"] else "N/A",
        "Version": parent["metadata"].get("version", "N/A") if parent["metadata"] else "N/A"
    })

parent_child_df = pd.DataFrame(parent_child_summary)

if len(parent_child_df) > 0:
    print("\n\n📊 Parent-Child Distribution (first 10):")
    print(parent_child_df.head(10).to_string(index=False))
    
    if len(parent_child_df) > 10:
        print(f"   ... and {len(parent_child_df) - 10} more parent chunks")

In [ ]:
# Investigate what the "regular chunks" are
print("\n🔍 Analysing Regular Chunks:")
print(f"Total regular chunks: {len(regular_chunks)}\n")

# Sample regular chunks
print("📋 Sample regular chunks (first 5):")
for i, chunk in enumerate(regular_chunks[:5]):
    chunk_id = chunk['id']
    metadata = chunk.get('metadata', {})
    print(f"\n  {i+1}. ID: {chunk_id}")
    print(f"     chunk_type: {metadata.get('chunk_type', 'N/A')}")
    print(f"     is_parent: {metadata.get('is_parent', 'N/A')}")
    print(f"     has parent_id: {'parent_id' in metadata}")
    
# Check if regular chunks match child chunk content
print("\n\n🔍 Checking for duplicate content:")
print("Comparing regular chunk IDs with child chunk patterns...")

regular_chunk_ids = [c['id'] for c in regular_chunks]
child_chunk_ids = [c['id'] for c in child_chunks]

print(f"\nRegular chunk ID pattern (first 3): {regular_chunk_ids[:3]}")
print(f"Child chunk ID pattern (first 3): {child_chunk_ids[:3]}")

# Check if they're storing the same content
print(f"\n⚠️  Potential issue: We may be storing child chunks TWICE:")
print(f"   - {len(regular_chunks)} as 'regular chunks' (from store_chunks_in_chroma)")
print(f"   - {len(child_chunks)} as 'child chunks' (from store_child_chunks)")
print(f"   Total storage: {len(regular_chunks) + len(child_chunks)} chunks for essentially the same content")